In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
from sklearn.model_selection import train_test_split

In [4]:
churn_train_data = pd.read_csv('../data/train.csv')
churn_test_data = pd.read_csv('../data/test.csv')
churn_val_data = pd.read_csv('../data/validation.csv')

In [5]:
overall_data = pd.concat([churn_train_data, churn_test_data, churn_val_data], ignore_index=True)

In [6]:
overall_data.to_csv('../data/overall_data.csv', index=False)

In [71]:
print("Churn value counts before dropna:")
print(overall_data['Churn'].value_counts())

Churn value counts before dropna:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [72]:
overall_data['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [73]:
overall_data.isnull().sum()

Age                                     0
Avg Monthly GB Download                 0
Avg Monthly Long Distance Charges       0
Churn Category                       5174
Churn Reason                         5174
Churn Score                             0
City                                    0
CLTV                                    0
Contract                                0
Country                                 0
Customer ID                             0
Customer Status                         0
Dependents                              0
Device Protection Plan                  0
Gender                                  0
Internet Service                        0
Internet Type                        1526
Lat Long                                0
Latitude                                0
Longitude                               0
Married                                 0
Monthly Charge                          0
Multiple Lines                          0
Number of Dependents              

In [74]:


# Fill NaN in Churn Category and Churn Reason with 'No Churn' or similar
overall_data['Churn Category'] = overall_data['Churn Category'].fillna('No Churn')
overall_data['Churn Reason'] = overall_data['Churn Reason'].fillna('No Churn')
overall_data.dropna(inplace=True)

In [75]:
overall_data = overall_data.drop(columns=['Customer ID'])

In [76]:
X,y = overall_data.drop(columns=['Churn']), overall_data['Churn']

In [77]:
overall_data['Churn'].value_counts()

Churn
0    1736
1     757
Name: count, dtype: int64

In [78]:
print("Data types:")
print(X.dtypes)
print("\nCategorical columns:")
categorical_cols = X.select_dtypes(include=['object', 'category']).columns
print(categorical_cols)
for col in categorical_cols:
    print(f"\n{col} unique values:")
    print(X[col].unique())

Data types:
Age                                    int64
Avg Monthly GB Download                int64
Avg Monthly Long Distance Charges    float64
Churn Category                           str
Churn Reason                             str
Churn Score                            int64
City                                     str
CLTV                                   int64
Contract                                 str
Country                                  str
Customer Status                          str
Dependents                             int64
Device Protection Plan                 int64
Gender                                   str
Internet Service                       int64
Internet Type                            str
Lat Long                                 str
Latitude                             float64
Longitude                            float64
Married                                int64
Monthly Charge                       float64
Multiple Lines                         int6

C:\Users\User\AppData\Local\Temp\ipykernel_18212\2282432491.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category']).columns


In [79]:
y.value_counts()

Churn
0    1736
1     757
Name: count, dtype: int64

In [80]:
a= X[['City','Country']]
a['City Count'] = X['City'].map(X['City'].value_counts())
print(a[a['City']=='Stockton'])

          City        Country  City Count
5     Stockton  United States          13
902   Stockton  United States          13
1628  Stockton  United States          13
2083  Stockton  United States          13
3184  Stockton  United States          13
3744  Stockton  United States          13
3903  Stockton  United States          13
4141  Stockton  United States          13
5298  Stockton  United States          13
5309  Stockton  United States          13
5579  Stockton  United States          13
6562  Stockton  United States          13
6815  Stockton  United States          13


In [81]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Drop columns with only 1 unique value as they provide no information
columns_to_drop = ['Country', 'Customer Status', 'Quarter', 'State','Churn Category',]
X = X.drop(columns=columns_to_drop)

# Define encoding strategies
# Ordinal for Contract (Month-to-Month < One Year < Two Year)
contract_mapping = {'Month-to-Month': 0, 'One Year': 1, 'Two Year': 2}
X['Contract'] = X['Contract'].map(contract_mapping)

# Binary encoding for Gender
le = LabelEncoder()
X['Gender'] = le.fit_transform(X['Gender'])

# OneHot for nominal with few categories
onehot_cols = ['Internet Type', 'Payment Method', 'Offer' ]
X = pd.get_dummies(X, columns=onehot_cols, drop_first=True)

# For high cardinality: Churn Reason (20 categories), City (427), Lat Long (592)
# Option 1: Frequency encoding
#X['Churn Reason Freq'] = X['Churn Reason'].map(X['Churn Reason'].value_counts())
X['City Freq'] = X['City'].map(X['City'].value_counts())
# Drop Lat Long as it's redundant with City and coordinates
X = X.drop(columns=['Lat Long', 'Churn Reason', 'City','Latitude','Zip Code'])

## Droping columns Satisfaction Score and Churn Score as they are highly correlated with Churn and may lead to data leakage
X = X.drop(columns=['Satisfaction Score', 'Churn Score'])


print("Encoded X shape:", X.shape)
print("Encoded X dtypes:")
print(X.dtypes)

Encoded X shape: (2493, 44)
Encoded X dtypes:
Age                                    int64
Avg Monthly GB Download                int64
Avg Monthly Long Distance Charges    float64
CLTV                                   int64
Contract                               int64
Dependents                             int64
Device Protection Plan                 int64
Gender                                 int64
Internet Service                       int64
Longitude                            float64
Married                                int64
Monthly Charge                       float64
Multiple Lines                         int64
Number of Dependents                   int64
Number of Referrals                    int64
Online Backup                          int64
Online Security                        int64
Paperless Billing                      int64
Partner                                int64
Phone Service                          int64
Population                             int64
Premium T

In [82]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [83]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)

c:\Users\User\OneDrive\Desktop\Customer Churn Prediction Automation\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [84]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.81      0.88      0.84       346
           1       0.67      0.52      0.59       153

    accuracy                           0.77       499
   macro avg       0.74      0.70      0.72       499
weighted avg       0.76      0.77      0.77       499



In [85]:
confusion_matrix(y_test, y_pred)

array([[306,  40],
       [ 73,  80]])

In [86]:
feature_importance = pd.Series(model.coef_[0], index=X.columns).sort_values(ascending=False)
print("Feature importance:")
print(feature_importance)

Feature importance:
Monthly Charge                       0.016007
Offer_Offer E                        0.013528
Paperless Billing                    0.010950
Age                                  0.010527
City Freq                            0.010010
Internet Type_Fiber Optic            0.009899
Longitude                            0.009689
Multiple Lines                       0.007139
Total Refunds                        0.006262
Streaming Music                      0.004984
Senior Citizen                       0.004411
Gender                               0.004243
Under 30                             0.003671
Payment Method_Mailed Check          0.003127
Streaming Movies                     0.002759
Total Charges                        0.002348
Total Long Distance Charges          0.002032
Offer_Offer B                        0.001994
Total Extra Data Charges             0.000250
Streaming TV                         0.000197
CLTV                                 0.000127
Population    

In [87]:
# Check correlation with target
correlations = X.corrwith(y).abs().sort_values(ascending=False)
print("Top correlations with Churn:")
print(correlations.head(10))

# Specifically check Churn Category and Churn Reason features
churn_cat_cols = [col for col in X.columns if 'Contract' in col]
print("\nChurn Category features:")
for col in churn_cat_cols:
    print(f"{col}: {X[col].sum()} positives")

print("\nChurn Reason Freq distribution:")
print(X.groupby(y)['Contract'].describe())

Top correlations with Churn:
Contract                       0.434354
Tenure in Months               0.413908
Offer_Offer E                  0.379178
Total Charges                  0.338268
Total Revenue                  0.337065
Number of Referrals            0.321591
Online Security                0.283490
Premium Tech Support           0.262590
Dependents                     0.256990
Total Long Distance Charges    0.255759
dtype: float64

Churn Category features:
Contract: 1718 positives

Churn Reason Freq distribution:
        count      mean       std  min  25%  50%  75%  max
Churn                                                     
0      1736.0  0.926267  0.848311  0.0  0.0  1.0  2.0  2.0
1       757.0  0.145310  0.421034  0.0  0.0  0.0  0.0  2.0


c:\Users\User\OneDrive\Desktop\Customer Churn Prediction Automation\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\User\OneDrive\Desktop\Customer Churn Prediction Automation\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [88]:
### Lets create a model with Random Forest and compare results with Logistic Regression
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42,class_weight='balanced')
rf_model.fit(X_train, y_train)
y_rf_pred = rf_model.predict(X_test)
print("Random Forest Classification Report:")
print(classification_report(y_test, y_rf_pred))


Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.93      0.88       346
           1       0.78      0.58      0.66       153

    accuracy                           0.82       499
   macro avg       0.81      0.75      0.77       499
weighted avg       0.82      0.82      0.81       499



In [89]:
rf_feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Random Forest Feature importance:")
print(rf_feature_importance.head(10))

Random Forest Feature importance:
Contract                       0.111331
Number of Referrals            0.067105
Tenure in Months               0.066928
Total Revenue                  0.061611
Total Charges                  0.056912
Monthly Charge                 0.052922
Total Long Distance Charges    0.044406
Age                            0.041546
CLTV                           0.039746
Longitude                      0.039354
dtype: float64


In [90]:
### Lets improve the recall as we want to catch as many churners as possible, even if it means more false positives
model_high_recall = LogisticRegression(max_iter=1000, class_weight='balanced')
model_high_recall.fit(X_train,y_train)
y_pred_high_recall = model_high_recall.predict(X_test)
print("High Recall Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_high_recall))

High Recall Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.69      0.77       346
           1       0.53      0.78      0.63       153

    accuracy                           0.72       499
   macro avg       0.70      0.74      0.70       499
weighted avg       0.77      0.72      0.73       499



c:\Users\User\OneDrive\Desktop\Customer Churn Prediction Automation\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [91]:
## I experimented with different models and adjusted the classification threshold to prioritise recall, 
# since missing churners has higher business cost than false positives.